> ## Análise da Taxa de Execução Orçamentária das Capitais (2020–2024)
>
> Este notebook usa os dados do FINBRA/Siconfi (Despesas por Função) para investigar como as 26 capitais brasileiras executam o orçamento público — ou seja, de tudo que é *empenhado* (comprometido), quanto realmente é *pago*.
>
> **Indicador principal:**
> `Taxa de Execução = (Despesas Pagas / Despesas Empenhadas) × 100`
> Uma taxa baixa sugere que parte do orçamento comprometido não saiu do caixa no ano, ficando como "restos a pagar" para o ano seguinte.
>
> **Pergunta norteadora:** existe diferença relevante entre o que as capitais prometem gastar e o que de fato pagam? Esse padrão é uniforme entre áreas (funções) ou concentrado em algumas?
>
> **Roteiro:**
> 1. Ranking de execução em Saúde e Educação
> 2. Identificação de um caso de destaque (Natal) e comparação com a média das capitais por função
> 3. Investigação das subfunções que explicam a queda
> 4. Verificação se o padrão é local (Natal) ou nacional (toda função crítica)





Configurando na raíz do projeto, para as determinadas execuções adiante.

Antes entre em Ambiente Venv no Python executando comando
e instalando requirements.txt

In [2]:
import sys
from pathlib import Path
import pandas as pd

# Adiciona a raiz do projeto ao sys.path
sys.path.insert(0, str(Path.cwd().parent))

from src.banco.conexao_duckdb import conectar
from src.utils.constantes import CAMINHO_DUCKDB

Passo 1 - conectar a duckdb pra acessar os dados do parquet atraves de uma view apontada.(tabela de consulta)
          em outras palavras , conectando-se ao banco de dados.

In [3]:
con = conectar(CAMINHO_DUCKDB)
print(f'Conectado a: {CAMINHO_DUCKDB}')

Conectado a: c:\Users\corre\OneDrive\Área de Trabalho\Desafio-Analista-de-Dados-Sefaz-Macei-\finbra.duckdb


Passo 2 - Utilizando consulta sql via DuckDB aqui estamos ranqueando
pela capital que de fato se comprometeu com a taxa de execução na função Saúde.
É realizado o mapeamento da capital na tabela.parquet , a soma dos empenhados e pagos para serem postos na fórmula após isso a execução e ordenação na tabela.

In [4]:
query_top8_saude = """
SELECT
    TRIM(
    regexp_extract(
        Instituição,
        'Prefeitura Municipal d[eo]\\s*(.*?)\\s*[-–]',
        1
    )
) AS Capital,

    SUM(CASE
        WHEN Coluna = 'Despesas Empenhadas'
        THEN Valor ELSE 0
    END) AS Empenhado,

    SUM(CASE
        WHEN Coluna = 'Despesas Pagas'
        THEN Valor ELSE 0
    END) AS Pago,

    ROUND(
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END)
        /
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END)
        * 100,
        2
    ) AS Taxa_Execucao

FROM despesas_finbra

WHERE ContaTipo = 'Função'
  AND Conta = '10 - Saúde'
  AND Ano BETWEEN 2020 AND 2024

GROUP BY Capital

ORDER BY Taxa_Execucao DESC

LIMIT 27
"""
df_taxa_execucao = con.execute(query_top8_saude).df()
df_taxa_execucao["Taxa_Execucao"] = df_taxa_execucao["Taxa_Execucao"].map(lambda x: f"{x:.2f}%")
df_taxa_execucao

,Capital,Empenhado,Pago,Taxa_Execucao
0,Recife,8.121497e+09,8.037495e+09,98.97%
1,Porto Velho,2.445924e+09,2.412778e+09,98.64%
2,Belém,5.824859e+09,5.744916e+09,98.63%
3,Maceió,5.039084e+09,4.933627e+09,97.91%
4,Goiânia,9.033624e+09,8.817437e+09,97.61%
5,Aracaju,2.918719e+09,2.825124e+09,96.79%
6,Salvador,1.123326e+10,1.086529e+10,96.72%
7,Curitiba,1.290027e+10,1.243334e+10,96.38%
8,Fortaleza,1.482427e+10,1.417669e+10,95.63%
9,Manaus,6.742050e+09,6.415444e+09,95.16%


In [ ]:
import matplotlib.pyplot as plt

df_plot = df_comparacao.sort_values("Diferenca")

cores = ["#d62728" if v < 0 else "#2ca02c" for v in df_plot["Diferenca"]]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(df_plot["Funcao"], df_plot["Diferenca"], color=cores)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Diferença na Taxa de Execução (p.p.) — Natal vs Média das Capitais")
ax.set_title("Onde Natal fica acima ou abaixo da média (2020–2024)")

for i, v in enumerate(df_plot["Diferenca"]):
    ax.text(v + (0.3 if v >= 0 else -0.3), i, f"{v:.1f}",
             va="center", ha="left" if v >= 0 else "right", fontsize=9)

plt.tight_layout()
plt.show()

NameError: name 'df_comparacao' is not defined

A análise da taxa de execução orçamentária na função Saúde (2020–2024) indica que Recife apresenta a maior eficiência entre as capitais analisadas, com 98,97%, seguida por Porto Velho (98,64%) e Belém (98,63%). Maceió aparece em quarto do Ranking com 97.91% indicando um alto nível de execução orçamentária, próximo das capitais mais eficientes do país. Natal aparece entre as menores taxas (~83,34%)


In [21]:
query_top8_educacao = """
SELECT
    TRIM(
    regexp_extract(
        Instituição,
        'Prefeitura Municipal d[eo]\\s*(.*?)\\s*[-–]',
        1
    )
) AS Capital,

    SUM(CASE
        WHEN Coluna = 'Despesas Empenhadas'
        THEN Valor ELSE 0
    END) AS Empenhado,

    SUM(CASE
        WHEN Coluna = 'Despesas Pagas'
        THEN Valor ELSE 0
    END) AS Pago,

    ROUND(
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END)
        /
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END)
        * 100,
        2
    ) AS Taxa_Execucao

FROM despesas_finbra

WHERE ContaTipo = 'Função'
  AND Conta = '12 - Educação'
  AND Ano BETWEEN 2020 AND 2024

GROUP BY Capital

ORDER BY Taxa_Execucao DESC

LIMIT 27
"""
df_taxa_execucao = con.execute(query_top8_educacao).df()
df_taxa_execucao["Taxa_Execucao"] = df_taxa_execucao["Taxa_Execucao"].map(lambda x: f"{x:.2f}%")
df_taxa_execucao

,Capital,Empenhado,Pago,Taxa_Execucao
0,Fortaleza,1.131096e+10,1.097499e+10,97.03%
1,Boa Vista,2.461227e+09,2.386931e+09,96.98%
2,Salvador,9.146231e+09,8.820142e+09,96.43%
3,Goiânia,7.201893e+09,6.925985e+09,96.17%
4,Porto Velho,2.262418e+09,2.168244e+09,95.84%
5,Aracaju,1.820513e+09,1.735163e+09,95.31%
6,Palmas,2.375581e+09,2.242823e+09,94.41%
7,Cuiabá,3.124880e+09,2.948428e+09,94.35%
8,Florianópolis,3.461265e+09,3.255235e+09,94.05%
9,Manaus,1.020885e+10,9.601847e+09,94.05%


Já na função Educação, observa-se uma mudança relevante no posicionamento de Maceió. A capital apresenta uma taxa de execução de 89,66%, ocupando uma posição intermediária no ranking, abaixo de capitais como Fortaleza (97,03%) e Salvador (96,43%), e também abaixo de sua própria performance na área de Saúde. Aqui Natal tambem aparece com menor taxa de 78.01%

Analisando esses resultados agora a pergunta que fica... Por que Natal apresenta menor taxa de execução orçamentária em comparação às demais capitais? Iremos olhar se isso é persistente então preciso saber se acontece em outras funções ou só em algumas áreas. Pra isso será necessário calcular:

Taxa de Execucao Natal por função (2020-2024)
Comparar com a média das capitais por função

In [25]:
query_natal = """
SELECT
    Conta AS Funcao,
    SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END) AS Pago,
    SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) AS Empenhado,
    CASE 
        WHEN SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) > 0
        THEN 
            (SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END)
            /
            SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END)) * 100
        ELSE NULL
    END AS Taxa_Execucao
FROM despesas_finbra
WHERE ContaTipo = 'Função'
  AND Ano BETWEEN 2020 AND 2024
  AND TRIM(
        regexp_extract(
            Instituição,
            'Prefeitura Municipal d[aeo]\\s*(.*?)\\s*[-–]',
            1
        )
      ) = 'Natal'
GROUP BY Conta
ORDER BY Taxa_Execucao DESC;
"""

df_natal = con.execute(query_natal).df()
df_natal["Taxa_Execucao"] = df_natal["Taxa_Execucao"].map(lambda x: f"{x:.2f}%")
df_natal

,Funcao,Pago,Empenhado,Taxa_Execucao
0,01 - Legislativa,4.622357e+08,4.649941e+08,99.41%
1,09 - Previdência Social,1.767225e+09,1.796631e+09,98.36%
2,06 - Segurança Pública,1.946548e+08,2.027738e+08,96.00%
3,03 - Essencial à Justiça,3.732450e+08,3.889101e+08,95.97%
4,08 - Assistência Social,3.781707e+08,4.024055e+08,93.98%
5,04 - Administração,4.504526e+08,4.852646e+08,92.83%
6,18 - Gestão Ambiental,2.392380e+07,2.590312e+07,92.36%
7,28 - Encargos Especiais,6.521633e+08,7.504766e+08,86.90%
8,27 - Desporto e Lazer,4.880790e+07,5.644722e+07,86.47%
9,10 - Saúde,4.844875e+09,5.813152e+09,83.34%


Ok agora iremos compara com a média de outras capitais:

In [4]:

query_comparacao="""
WITH base AS (
    SELECT
        TRIM(
            regexp_extract(Instituição, 'Prefeitura Municipal d[eo]\\s*(.*?)\\s*[-–]', 1)
        ) AS Capital,
        Conta AS Funcao,
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) AS Empenhado,
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END) AS Pago
    FROM despesas_finbra
    WHERE ContaTipo = 'Função'
      AND Ano BETWEEN 2020 AND 2024
    GROUP BY Capital, Funcao
),
taxas AS (
    SELECT
        Capital,
        Funcao,
        Empenhado,
        Pago,
        CASE 
            WHEN Empenhado > 0 THEN (Pago / Empenhado) * 100
            ELSE NULL
        END AS Taxa_Execucao
    FROM base
),
media_capitais AS (
    SELECT
        Funcao,
        AVG(Taxa_Execucao) AS Media_Funcao
    FROM taxas
    GROUP BY Funcao
)
SELECT
    t.Funcao,
    t.Capital,
    ROUND(t.Taxa_Execucao, 2) AS Taxa_Execucao_Natal,
    ROUND(m.Media_Funcao, 2) AS Media_Capitais,
    ROUND(t.Taxa_Execucao - m.Media_Funcao, 2) AS Diferenca
FROM taxas t
JOIN media_capitais m
    ON t.Funcao = m.Funcao
WHERE t.Capital = 'Natal'
ORDER BY Diferenca ASC;
"""

df_comparacao = con.execute(query_comparacao).df()
df_comparacao

NameError: name 'con' is not defined

In [3]:
import matplotlib.pyplot as plt

df_plot = df_comparacao.sort_values("Diferenca")

cores = ["#d62728" if v < 0 else "#2ca02c" for v in df_plot["Diferenca"]]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(df_plot["Funcao"], df_plot["Diferenca"], color=cores)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Diferença na Taxa de Execução (p.p.) — Natal vs Média das Capitais")
ax.set_title("Onde Natal fica acima ou abaixo da média (2020–2024)")

for i, v in enumerate(df_plot["Diferenca"]):
    ax.text(v + (0.3 if v >= 0 else -0.3), i, f"{v:.1f}",
             va="center", ha="left" if v >= 0 else "right", fontsize=9)

plt.tight_layout()
plt.show()

NameError: name 'df_comparacao' is not defined

A comparação entre Natal vs média das capitais (2020–2024) mostra um padrão claro de subexecução orçamentária em múltiplas funções. Natal está abaixo da média em 9 de 15 funções analisadas. As maiores quedas sendo em áreas estruturais como Habitação menos 19,12 pontos percentuais abaixo da média, logo em seguida Urbanismo com menos 12,43 abaixo. Isso indica que o problema não é isolado, é sistêmico em áreas de investimento e políticas públicas estruturantes. 

Ou seja, Natal não apresenta um problema geral uniforme de execução, mas sim um padrão concentrado em funções de investimento e desenvolvimento urbano/social

Com isso,  Por que essas funções puxam Natal para baixo?
Será necessário investigar dentro de suas subfunções. Pra isso vamos nos aprofundar nas funções mais críticas para encontrar determinados padrões em suas subfunções.

In [30]:
query_subfuncoes = """
WITH base AS (
    SELECT
        TRIM(regexp_extract(Instituição, 'Prefeitura Municipal d[eo]\\s*(.*?)\\s*[-–]', 1)) AS Capital,
        Conta AS Subfuncao,
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) AS Empenhado,
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END) AS Pago
    FROM despesas_finbra
    WHERE ContaTipo = 'Subfunção'
      AND Ano BETWEEN 2020 AND 2024
      AND TRIM(regexp_extract(Instituição, 'Prefeitura Municipal d[eo]\\s*(.*?)\\s*[-–]', 1)) = 'Natal'
    GROUP BY Capital, Subfuncao
),
taxas AS (
    SELECT
        Capital,
        Subfuncao,
        Empenhado,
        Pago,
        CASE 
            WHEN Empenhado > 0 THEN (Pago / Empenhado) * 100
            ELSE NULL
        END AS Taxa_Execucao
    FROM base
)
SELECT
    Capital,
    Subfuncao,
    Empenhado,
    Pago,
    ROUND(Taxa_Execucao, 2) AS Taxa_Execucao
FROM taxas
ORDER BY Taxa_Execucao ASC;
"""
df_sub = con.execute(query_subfuncoes).df()
df_sub

,Capital,Subfuncao,Empenhado,Pago,Taxa_Execucao
0,Natal,16.482 - Habitação Urbana,1.118873e+07,2.458173e+06,21.97
1,Natal,15.451 - Infraestrutura Urbana,5.108514e+08,2.097684e+08,41.06
2,Natal,"13.391 - Patrimônio Histórico, Artístico e Arq...",6.531600e+05,3.172600e+05,48.57
3,Natal,08.241 - Assistência ao Idoso,1.033968e+06,5.657184e+05,54.71
4,Natal,10.303 - Suporte Profilático e Terapêutico,1.097963e+08,6.264250e+07,57.05
5,Natal,15.453 - Transportes Coletivos Urbanos,2.407342e+08,1.473069e+08,61.19
6,Natal,27.813 - Lazer,8.837322e+06,5.804856e+06,65.69
7,Natal,04.126 - Tecnologia da Informação,7.849043e+06,5.390250e+06,68.67
8,Natal,13.392 - Difusão Cultural,1.955127e+08,1.399872e+08,71.60
9,Natal,12.122 - Administração Geral,3.165837e+08,2.272389e+08,71.78


A análise da taxa de execução orçamentária por função no período de 2020 a 2024 para a capital Natal revela um padrão relevante de heterogeneidade entre áreas de atuação. Observa-se que funções como Habitação Urbana (21,97%) e Infraestrutura Urbana (41,06%) apresentam baixa taxa de execução, indicando forte discrepância entre o orçamento empenhado e o efetivamente pago.

Em contrapartida, áreas como Saúde, Educação e Previdência apresentam taxas significativamente mais elevadas, acima de 78%, sugerindo maior regularidade na execução orçamentária.

Esse comportamento indica que a baixa taxa de execução global de Natal não é generalizada, mas concentrada em funções de investimento urbano e estrutural, enquanto áreas de manutenção de serviços essenciais apresentam desempenho mais estável

Porém isso deve ser verificado se é um problema específico de Natal ou um padrão nacional dessas funções? Pra isso deve ser verificado como é taxa de execução dessas funções nas capitais e comparar com Natal.

In [ ]:
query_func_critical = """
WITH base AS (
    SELECT
        TRIM(
            regexp_extract(Instituição, 'Prefeitura Municipal d[eo]\\s*(.*?)\\s*[-–]', 1)
        ) AS Capital,
        Conta AS Funcao,
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) AS Empenhado,
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END) AS Pago
    FROM despesas_finbra
    WHERE ContaTipo = 'Função'
      AND Ano BETWEEN 2020 AND 2024
      AND Conta IN (
          '16 - Habitação',
          '15 - Urbanismo'
      )
    GROUP BY Capital, Funcao
),
taxas AS (
    SELECT
        Capital,
        Funcao,
        Empenhado,
        Pago,
        CASE 
            WHEN Empenhado > 0 THEN (Pago / Empenhado) * 100
            ELSE NULL
        END AS Taxa_Execucao
    FROM base
)
SELECT
    Capital,
    Funcao,
    ROUND(Empenhado, 2) AS Empenhado,
    ROUND(Pago, 2) AS Pago,
    ROUND(Taxa_Execucao, 2) AS Taxa_Execucao
FROM taxas
ORDER BY Funcao, Taxa_Execucao ASC;
"""  

df_sub_critical = con.execute(query_func_critical).df()
df_sub_critical

,Capital,Funcao,Empenhado,Pago,Taxa_Execucao
0,Curitiba,15 - Urbanismo,7.291745e+09,4.968142e+09,68.13
1,Macapá,15 - Urbanismo,1.287650e+09,9.363065e+08,72.71
2,Natal,15 - Urbanismo,3.197655e+09,2.360973e+09,73.83
3,Porto Alegre,15 - Urbanismo,1.296664e+09,1.017608e+09,78.48
4,Porto Velho,15 - Urbanismo,1.215666e+09,9.690080e+08,79.71
5,Maceió,15 - Urbanismo,1.851700e+09,1.489622e+09,80.45
6,São Luís,15 - Urbanismo,1.781319e+09,1.437328e+09,80.69
7,São Paulo,15 - Urbanismo,4.466983e+10,3.613530e+10,80.89
8,Vitória,15 - Urbanismo,1.709852e+09,1.403734e+09,82.10
9,Belo Horizonte,15 - Urbanismo,3.710946e+09,3.157808e+09,85.09


In [34]:
query_resumo_func = """
WITH base AS (
    SELECT
        TRIM(
            regexp_extract(Instituição, 'Prefeitura Municipal d[eo]\\s*(.*?)\\s*[-–]', 1)
        ) AS Capital,
        Conta AS Funcao,
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) AS Empenhado,
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END) AS Pago
    FROM despesas_finbra
    WHERE ContaTipo = 'Função'
      AND Ano BETWEEN 2020 AND 2024
      AND Conta IN ('16 - Habitação', '15 - Urbanismo')
    GROUP BY Capital, Funcao
),
taxas AS (
    SELECT
        Capital,
        Funcao,
        (Pago / NULLIF(Empenhado, 0)) * 100 AS Taxa_Execucao
    FROM base
)
SELECT
    Funcao,
    ROUND(AVG(Taxa_Execucao), 2) AS Media_Nacional,
    ROUND(MIN(Taxa_Execucao), 2) AS Min_Taxa,
    ROUND(MAX(Taxa_Execucao), 2) AS Max_Taxa
FROM taxas
GROUP BY Funcao
ORDER BY Media_Nacional ASC;
"""

df_resumido = con.execute(query_resumo_func).df()
df_resumido

,Funcao,Media_Nacional,Min_Taxa,Max_Taxa
0,16 - Habitação,84.99,54.78,100.00
1,15 - Urbanismo,86.27,68.13,97.14


A análise por subfunção indica que Habitação e Urbanismo apresentam elevada dispersão entre capitais, com diferenças significativas entre os maiores e menores níveis de execução.

Habitação, em particular, apresenta o maior intervalo observado (54,78% a 100%), sugerindo forte heterogeneidade na capacidade de execução entre municípios.

No caso de Natal, as taxas observadas nessas funções situam-se abaixo da média nacional, o que contribui para seu desempenho relativo inferior. No entanto, o padrão indica que essas áreas são estruturalmente mais instáveis em nível nacional, não sendo um problema exclusivo do município

Para entender se a baixa taxa de execução em Habitação e Urbanismo é um problema específico de Natal ou um padrão nacional dessas funções, será necessário comparar o comportamento dessas mesmas funções em todas as capitais. Dessa forma responde-se Natal está abaixo da média por uma falha local de execução orçamentária ou Se essas funções possuem, em geral, baixa execução em todo o país. Pra isso será calculado  Taxa de execução média por função (Habitação e Urbanismo) em todas as capitais,  Comparação direta entre Natal e a média nacional dessas funções ,  Identificação de posição relativa de Natal no ranking dessas funções.


In [7]:
query_natalc = """
WITH base AS (
    SELECT
        TRIM(
            regexp_extract(Instituição, 'Prefeitura Municipal d[eo]\\s*(.*?)\\s*[-–]', 1)
        ) AS Capital,
        Conta AS Funcao,
        SUM(CASE WHEN Coluna = 'Despesas Empenhadas' THEN Valor ELSE 0 END) AS Empenhado,
        SUM(CASE WHEN Coluna = 'Despesas Pagas' THEN Valor ELSE 0 END) AS Pago
    FROM despesas_finbra
    WHERE ContaTipo = 'Função'
      AND Ano BETWEEN 2020 AND 2024
      AND Conta IN ('15 - Urbanismo', '16 - Habitação')
    GROUP BY Capital, Funcao
),
taxas AS (
    SELECT
        Capital,
        Funcao,
        (Pago / NULLIF(Empenhado,0)) * 100 AS Taxa_Execucao
    FROM base
),
ranking AS (
    SELECT
        Capital,
        Funcao,
        Taxa_Execucao,
        AVG(Taxa_Execucao) OVER (PARTITION BY Funcao) AS Media_Nacional,
        RANK() OVER (PARTITION BY Funcao ORDER BY Taxa_Execucao DESC) AS Rank_Funcao
    FROM taxas
)
SELECT *
FROM ranking
WHERE Capital IN ('Natal')
   OR Rank_Funcao <= 5
ORDER BY Funcao, Rank_Funcao;
"""

df_natalc = con.execute(query_natalc).df()
df_natalc

NameError: name 'con' is not defined

A análise comparativa entre Natal e a média das capitais nas funções de Urbanismo e Habitação indica que o baixo desempenho de Natal não é um padrão estrutural dessas funções no Brasil.Observa-se que, em ambas as funções, a maioria das capitais apresenta taxas de execução elevadas, frequentemente acima de 90%, com algumas chegando próximas de 100%. No entanto, Natal aparece de forma consistente entre os piores desempenhos do ranking, com Urbanismo: 73,83% e Habitação: 65,87%. Esse comportamento contrasta fortemente com a média nacional, que se mantém em patamares significativamente superiores (≈86% em Urbanismo e ≈85% em Habitação). 

Isso indica que o problema não está nas funções em si, mas sim na capacidade de execução orçamentária específica de Natal nessas áreas.
As evidências sugerem que as funções de Urbanismo e Habitação são os principais vetores que puxam a taxa geral de execução de Natal para baixo, possivelmente associadas a gargalos locais de implementação, execução de obras ou baixa conversão de orçamento empenhado em pagamento efetivo.

In [8]:
con.close()

NameError: name 'con' is not defined